# Choosing the Right Kernel: A Hands-On SVM Tutorial

**GitHub:** https://github.com/VineethDoddaballapuraRaju24089093/ml_assinment_2  
**License:** MIT

---

This notebook walks through how different SVM kernel functions : Linear, Polynomial, and RBF.
affect the decision boundary and test performance, using the UCI Wine Recognition dataset.
It then goes further: we derive what actually *makes* a function a valid kernel (Mercer's condition)
and build a custom Laplacian kernel from scratch using only numpy.

**By the end of this notebook you will be able to:**
- Explain what a support vector machine does and why the margin matters
- Describe the kernel trick and when to use Linear, Polynomial, or RBF kernels
- Tune the C and gamma hyperparameters using GridSearchCV
- State Mercer's condition and verify it numerically
- Implement a custom kernel using scikit-learn's `kernel='precomputed'` API

### Contents
1. [Setup & Imports](#1-setup--imports)
2. [The core idea: margin and support vectors](#2-the-core-idea-margin-and-support-vectors) → *Figure 1*
3. [The kernel trick — what it is and why it works](#3-the-kernel-trick)
4. [Seeing kernels in action (Moons dataset)](#4-seeing-kernels-in-action) → *Figure 2*
5. [The Wine dataset](#5-the-wine-dataset) → *Figure 3*
6. [Preprocessing — this step really matters](#6-preprocessing-this-step-really-matters)
7. [Fitting SVMs with different kernels](#7-fitting-svms-with-different-kernels)
8. [Comparing accuracy](#8-accuracy-comparison) → *Figure 4*
9. [The C and gamma landscape (RBF heatmap)](#9-the-c-and-gamma-landscape) → *Figure 5*
10. [Doing it properly with GridSearchCV](#10-doing-it-properly-with-gridsearchcv)
11. [Final evaluation](#11-final-evaluation) → *Figures 6 & 7*
12. [What makes a valid kernel? Mercer's condition](#12-what-makes-a-valid-kernel-mercers-condition)
13. [Building a custom Laplacian kernel from scratch](#13-building-a-custom-kernel-from-scratch) → *Figure 8*
14. [What I learned](#14-what-i-learned)

### Figure Index
| Figure | Description | Section |
|--------|-------------|---------|
| Fig 1 | Decision boundary, margin & support vectors (toy data) | s2 |
| Fig 2 | Decision boundaries for three kernels on the Moons dataset | s4 |
| Fig 3 | Wine dataset scatter: Alcohol vs Flavanoids | s5 |
| Fig 4 | Train/test accuracy comparison for five kernel configurations | s8 |
| Fig 5 | RBF accuracy heatmap: C vs gamma | s9 |
| Fig 6 | What happens if you forget to standardise (demo) | s6 |
| Fig 7 | All kernels compared including custom Laplacian | s13 |
| Fig 8 | Confusion matrix for the best-tuned RBF SVM | s11 |

> **Accessibility:** All figures use a colourblind-friendly palette (Wong, 2011) with distinct
> point shapes so colour is never the only channel carrying information. The heatmap (Fig 5)
> uses the *cividis* colormap (Nuñez et al., 2018). Markdown cells use semantic H2/H3 headings
> for screen-reader compatibility.


## 1. Setup & Imports

Standard imports plus a colourblind-friendly palette (Wong, 2011) used throughout
so anyone can read these plots regardless of colour vision.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

from sklearn import svm
from sklearn.datasets import load_wine, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Create figures folder notebook saves plots here
os.makedirs('figures', exist_ok=True)

# Colourblind-friendly palette (Wong 2011)
# Works for deuteranopia and protanopia
CB = {
    'blue':   '#0072B2',
    'orange': '#E69F00',
    'green':  '#009E73',
    'red':    '#D55E00',
    'sky':    '#56B4E9',
    'black':  '#000000',
}
print('All good libraries loaded.')


## 2. The Core Idea: Margin and Support Vectors

An SVM doesn't just find *a* boundary between two classes. it finds the one with the
*widest possible gap* (the **margin**). The points sitting right on the edge of that gap
are the **support vectors**. Everything else in the training set is irrelevant once
training is done.

Let's build that from scratch on a small toy dataset so we can actually see it.
*(Figure 1 below)*


In [ ]:
np.random.seed(42)
X_toy = np.r_[np.random.randn(20,2)-[2,2], np.random.randn(20,2)+[2,2]]
y_toy = np.array([0]*20 + [1]*20)

clf_lin = svm.SVC(kernel='linear', C=1)
clf_lin.fit(X_toy, y_toy)

# Draw the decision boundary manually from the weight vector
w = clf_lin.coef_[0]
b = clf_lin.intercept_[0]
xx = np.linspace(-5, 5, 200)
yy = (-w[0]*xx - b) / w[1]
margin = 1 / np.linalg.norm(w)
yy_up   = yy + np.sqrt(1 + (w[0]/w[1])**2) * margin
yy_down = yy - np.sqrt(1 + (w[0]/w[1])**2) * margin

fig, ax = plt.subplots(figsize=(6,5))
colors_toy = [CB['blue'] if yi==0 else CB['orange'] for yi in y_toy]
ax.scatter(X_toy[:,0], X_toy[:,1], c=colors_toy, s=60, zorder=3, edgecolors='k', lw=0.5)
ax.plot(xx, yy,      color=CB['black'], lw=2,   label='Decision boundary')
ax.plot(xx, yy_up,   color=CB['red'],   lw=1.5, linestyle='--', label='Margin edge')
ax.plot(xx, yy_down, color=CB['red'],   lw=1.5, linestyle='--')
ax.scatter(clf_lin.support_vectors_[:,0], clf_lin.support_vectors_[:,1],
           s=200, facecolors='none', edgecolors=CB['red'], lw=2, label='Support vectors')
p0 = mpatches.Patch(color=CB['blue'],   label='Class 0')
p1 = mpatches.Patch(color=CB['orange'], label='Class 1')
h, l = ax.get_legend_handles_labels()
ax.legend(handles=[p0, p1]+h, fontsize=9)
ax.set_xlim(-5,5); ax.set_ylim(-5,5)
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
ax.set_title('Figure 1 — SVM: Decision Boundary, Margin & Support Vectors', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig1_margin.png', dpi=150)
plt.show()

print(f'Support vectors per class: {clf_lin.n_support_}')
print('Notice: only a handful of points define the entire boundary.')
print()
print('Figure 1 alt-text: Scatter plot of two groups of points (blue circles = Class 0,')
print('orange circles = Class 1) separated by a solid black decision boundary. Two red')
print('dashed lines show the margin edges. Red open circles mark the support vectors.')


## 3. The Kernel Trick

Here's the problem: most interesting datasets *aren't* linearly separable.
So how do we use an SVM on them?

The answer is the **kernel trick**. Instead of explicitly transforming data into a
higher-dimensional space (which would be computationally expensive), a kernel function
K(x, x') computes the *inner product* in that space directly. The SVM never needs to
know what the space looks like. it just needs K.

| Kernel | Formula | What it does |
|--------|---------|-------------|
| Linear | K(x, x') = x · x' | No transformation. Best for already-separable or high-dim data. |
| Polynomial | K(x, x') = (γ x·x' + r)^d | Captures feature interactions up to degree d. |
| RBF | K(x, x') = exp(−γ ‖x−x'‖²) | Similarity by distance. Most flexible, best default. |

Two hyperparameters matter most:
- **C**: regularisation strength. Small C = wide margin, allows some errors. Large C = tight fit, risk of overfitting.
- **gamma (γ)**: RBF and Polynomial only. How far each point's influence reaches. High γ → wiggly, over-local boundary. Low γ → smooth, more generalised boundary.

**Scikit-learn note:** When classifying more than two classes, scikit-learn's `SVC`
uses a one-vs-one strategy by default (one binary classifier per pair of classes).
For the Wine dataset with 3 classes this means 3 binary classifiers are combined.


## 4. Seeing Kernels in Action

Before touching the real dataset, it's worth *seeing* what each kernel does to a decision
boundary. The Moons dataset is perfect for this two interleaved half-circles that no
straight line can separate. *(Figure 2 below)*


In [ ]:
X_m, y_m = make_moons(n_samples=200, noise=0.2, random_state=42)
sc_m = StandardScaler()
X_ms = sc_m.fit_transform(X_m)

kernels_cfg = [
    ('Linear',           dict(kernel='linear', C=1)),
    ('Polynomial (d=3)', dict(kernel='poly', C=1, degree=3, coef0=1)),
    ('RBF',              dict(kernel='rbf',  C=1, gamma='scale')),
]

xx_m, yy_m = np.meshgrid(np.linspace(-2.5,3,300), np.linspace(-1.5,2.5,300))
grid_m = np.c_[xx_m.ravel(), yy_m.ravel()]
cmap_bg = ListedColormap([CB['sky'], CB['orange']])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (title, params) in zip(axes, kernels_cfg):
    clf = svm.SVC(**params)
    clf.fit(X_ms, y_m)
    Z = clf.predict(grid_m).reshape(xx_m.shape)
    ax.contourf(xx_m, yy_m, Z, alpha=0.3, cmap=cmap_bg)
    ax.contour( xx_m, yy_m, Z, colors='k', linewidths=0.8)
    ax.scatter(X_ms[y_m==0,0], X_ms[y_m==0,1], c=CB['blue'],   marker='o',
               edgecolors='k', lw=0.4, s=30, label='Class 0')
    ax.scatter(X_ms[y_m==1,0], X_ms[y_m==1,1], c=CB['orange'], marker='s',
               edgecolors='k', lw=0.4, s=30, label='Class 1')
    ax.set_title(f'{title}\nTrain acc = {clf.score(X_ms, y_m):.2f}', fontweight='bold')
    ax.legend(fontsize=8)
fig.suptitle('Figure 2 — Decision boundaries: three kernels on the Moons dataset',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/fig2_kernels_boundary.png', dpi=150, bbox_inches='tight')
plt.show()

print('The linear kernel draws a straight line and gets it badly wrong.')
print('RBF finds a natural, smooth boundary. Polynomial is somewhere in between.')
print()
print('Figure 2 alt-text: Three side-by-side scatter plots showing two classes (blue circles')
print('and orange squares) on the Moons dataset. Left panel (Linear): a diagonal straight-line')
print('boundary misclassifies many points. Middle panel (Polynomial d=3): a curved boundary')
print('fits better. Right panel (RBF): a smooth boundary follows the crescent shapes closely.')


## 5. The Wine Dataset

Now for the real data. The UCI Wine Recognition Dataset (Aeberhard, Coomans and de Vel, 1992)
contains 178 Italian wine samples from three cultivars, each described by 13 chemical
measurements.

I picked this dataset because it's genuinely interesting, it's clean (no missing values),
and it hasn't been used in every SVM tutorial ever written. It's also a realistic challenge.
the classes are separable but not trivially so. *(Figure 3 below)*

> **Reference:** Aeberhard, S. et al. (1992). Wine Data Set. UCI ML Repository.
> https://archive.ics.uci.edu/ml/datasets/wine


In [ ]:
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['cultivar'] = wine.target

print('Dataset shape:', df.shape)
print('Class names:', wine.target_names)
print('\nSamples per class:')
print(df['cultivar'].value_counts().sort_index())
df.describe().round(2)


In [ ]:
# Plot two of the most visually separable features
fig, ax = plt.subplots(figsize=(7,5))
markers = ['o', 's', '^']
pals    = [CB['blue'], CB['orange'], CB['green']]
for cls, color, marker in zip([0,1,2], pals, markers):
    mask = df['cultivar'] == cls
    ax.scatter(df.loc[mask,'alcohol'], df.loc[mask,'flavanoids'],
               c=color, marker=marker, s=55, edgecolors='k', lw=0.4,
               label=f'{wine.target_names[cls]}', alpha=0.85)
ax.set_xlabel('Alcohol (%)', fontsize=12)
ax.set_ylabel('Flavanoids', fontsize=12)
ax.set_title('Figure 3 — Wine Dataset: Alcohol vs Flavanoids by Cultivar', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('figures/fig3_wine_scatter.png', dpi=150)
plt.show()

print('Figure 3 alt-text: Scatter plot with three groups distinguished by both colour and shape.')
print('  Blue circles = class_0 (Barolo), orange squares = class_1 (Grignolino),')
print('  green triangles = class_2 (Barbera). X-axis: Alcohol (%). Y-axis: Flavanoids.')
print('  The three groups show partial separation — not perfectly clean, which makes')
print('  this a meaningful test case for kernel comparison.')


## 6. Preprocessing This Step Really Matters

I cannot stress this enough: **always standardise your features before fitting an SVM.**

SVMs work by maximising a margin defined in terms of Euclidean distances.
If one feature has values ranging from 0 to 1000 and another from 0 to 1,
the first feature will completely drown out the second. `StandardScaler` fixes this
by transforming every feature to zero mean and unit variance.

> ⚠️ **Important:** Fit the scaler on training data only. Never on the full dataset.
> Doing so leaks test information into training and gives you a falsely optimistic accuracy.

**Try it yourself : what happens if you skip standardisation?**  
The cell below (Figure 6) runs an RBF SVM on *raw* (unscaled) features and compares it
to the scaled version. Run it to see the damage.


In [ ]:
X = wine.data
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # fit on train only
X_test_s  = scaler.transform(X_test)       # apply same transform to test

print(f'Train: {X_train_s.shape[0]} samples | Test: {X_test_s.shape[0]} samples')
print(f'Training feature means (should be ~0): {X_train_s.mean(axis=0).round(2)}')


In [ ]:
# ── Try it yourself: what happens without standardisation? ───────────────
# Run this cell AFTER the scaler cell above (Section 6) to compare
# raw vs scaled features. This is an intermediate step not shown in the PDF.

X_train_raw, X_test_raw = train_test_split(
    wine.data, test_size=0.25, random_state=42, stratify=wine.target
)[::1]  # just the X splits

# Re-split cleanly to get both raw X and raw y
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    wine.data, wine.target, test_size=0.25, random_state=42, stratify=wine.target
)

clf_unscaled = svm.SVC(kernel='rbf', C=1, gamma='scale')
clf_scaled   = svm.SVC(kernel='rbf', C=1, gamma='scale')

clf_unscaled.fit(X_train_raw, y_train_raw)
clf_scaled.fit(X_train_s,     y_train)       # X_train_s from the cell above

acc_unscaled_train = clf_unscaled.score(X_train_raw, y_train_raw)
acc_unscaled_test  = clf_unscaled.score(X_test_raw,  y_test_raw)
acc_scaled_train   = clf_scaled.score(X_train_s,     y_train)
acc_scaled_test    = clf_scaled.score(X_test_s,      y_test)

x = np.arange(2)
width = 0.35
fig, ax = plt.subplots(figsize=(6, 4))
b1 = ax.bar(x - width/2, [acc_unscaled_train, acc_scaled_train], width,
            label='Train', color=CB['blue'],   edgecolor='k', lw=0.6)
b2 = ax.bar(x + width/2, [acc_unscaled_test,  acc_scaled_test],  width,
            label='Test',  color=CB['orange'], edgecolor='k', lw=0.6)
ax.set_xticks(x)
ax.set_xticklabels(['Unscaled\n(raw features)', 'Scaled\n(StandardScaler)'], fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Figure 6 — Effect of Standardisation on RBF SVM', fontweight='bold')
ax.legend(fontsize=11)
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('figures/fig6_standardisation_effect.png', dpi=150)
plt.show()

print(f'Without scaling  — train: {acc_unscaled_train:.3f}  test: {acc_unscaled_test:.3f}')
print(f'With scaling     — train: {acc_scaled_train:.3f}  test: {acc_scaled_test:.3f}')
print()
print('Figure 6 alt-text: Grouped bar chart comparing RBF SVM accuracy with and without')
print('StandardScaler. Left pair (Unscaled) shows substantially lower test accuracy.')
print('Right pair (Scaled) shows higher and more consistent train/test accuracy,')
print('demonstrating why standardisation is mandatory before fitting an SVM.')


## 7. Fitting SVMs With Different Kernels

Let's now fit five configurations and compare them side by side.
The results feed into Figure 4 (accuracy comparison) and Figure 7 (all kernels including
the custom Laplacian, built in section 13).


In [ ]:
configs = {
    'Linear':        svm.SVC(kernel='linear', C=1),
    'Poly (d=2)':    svm.SVC(kernel='poly', degree=2, C=1, coef0=1),
    'Poly (d=3)':    svm.SVC(kernel='poly', degree=3, C=1, coef0=1),
    'RBF (default)': svm.SVC(kernel='rbf',  C=1, gamma='scale'),
    'RBF (tuned)':   svm.SVC(kernel='rbf',  C=10, gamma=0.01),
}

results = []
for name, clf in configs.items():
    clf.fit(X_train_s, y_train)
    tr = clf.score(X_train_s, y_train)
    te = clf.score(X_test_s,  y_test)
    results.append({'Configuration': name, 'Train acc': round(tr,3), 'Test acc': round(te,3)})
    print(f'{name:20s}  train={tr:.3f}  test={te:.3f}')

results_df = pd.DataFrame(results)


## 8. Accuracy Comparison

*(Figure 4)* A bar chart makes the train/test gap visible at a glance a large gap
signals overfitting. Note that the linear kernel is surprisingly competitive: on
13-dimensional standardised data the three cultivars are nearly linearly separable.


In [ ]:
x = np.arange(len(results_df))
width = 0.35
fig, ax = plt.subplots(figsize=(9,5))
bars1 = ax.bar(x-width/2, results_df['Train acc'], width,
               label='Training accuracy', color=CB['blue'],   edgecolor='k', lw=0.6)
bars2 = ax.bar(x+width/2, results_df['Test acc'],  width,
               label='Test accuracy',     color=CB['orange'], edgecolor='k', lw=0.6)
ax.set_xticks(x)
ax.set_xticklabels(results_df['Configuration'], fontsize=10)
ax.set_ylim(0.7, 1.05)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Figure 4 — Kernel Accuracy Comparison: Wine Dataset', fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('figures/fig4_accuracy_comparison.png', dpi=150)
plt.show()

print('Figure 4 alt-text: Grouped bar chart with five kernel configurations on the x-axis.')
print('Blue bars = training accuracy, orange bars = test accuracy. Values labelled on each bar.')
print('Linear SVM achieves ~0.98 test accuracy; Poly d=3 shows a small train/test gap;')
print('RBF default is lower, but RBF tuned matches the linear result.')


## 9. The C and Gamma Landscape

*(Figure 5)* The RBF kernel's two hyperparameters (C and gamma) interact in non-obvious ways.
Sweeping over a grid reveals two clear failure modes:

- **High gamma** (bottom rows) → the model becomes hyper-local, essentially memorising
  training points. Test accuracy collapses regardless of C.
- **Very low C with moderate gamma** → the margin is too wide; the model underfits.

The best performance sits in the top-right region: moderate-to-high C with low gamma.
In practice you'd find this automatically with `GridSearchCV` (Section 10), but the
visual helps you *understand* what you're searching for.


In [ ]:
C_vals     = [0.1, 1, 10, 100]
gamma_vals = [0.001, 0.01, 0.1, 1]
acc_grid   = np.zeros((len(gamma_vals), len(C_vals)))

for i, g in enumerate(gamma_vals):
    for j, c in enumerate(C_vals):
        clf_tmp = svm.SVC(kernel='rbf', C=c, gamma=g)
        clf_tmp.fit(X_train_s, y_train)
        acc_grid[i, j] = clf_tmp.score(X_test_s, y_test)

# cividis is a colourblind-safe sequential colormap (Nuñez et al., 2018)
# It is perceptually uniform and readable under deuteranopia and protanopia.
fig, ax = plt.subplots(figsize=(7,5))
im = ax.imshow(acc_grid, cmap='cividis', vmin=0.6, vmax=1.0)
ax.set_xticks(range(len(C_vals)))
ax.set_xticklabels([str(c) for c in C_vals], fontsize=11)
ax.set_yticks(range(len(gamma_vals)))
ax.set_yticklabels([str(g) for g in gamma_vals], fontsize=11)
ax.set_xlabel('C (regularisation)', fontsize=12)
ax.set_ylabel('Gamma', fontsize=12)
ax.set_title('Figure 5 — RBF SVM Test Accuracy: C vs Gamma (Wine Dataset)', fontweight='bold')
cbar = plt.colorbar(im, ax=ax, label='Test Accuracy')
# Text colour: white on dark cells, black on light cells
for i in range(len(gamma_vals)):
    for j in range(len(C_vals)):
        val = acc_grid[i, j]
        txt_color = 'white' if val < 0.82 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=12, color=txt_color, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig5_rbf_heatmap.png', dpi=150)
plt.show()

print('High gamma (bottom rows) = overfitting regardless of C.')
print('Best results: low-moderate gamma, moderate-high C.')
print()
print('Figure 5 alt-text: 4x4 heatmap grid. Rows (top to bottom): gamma = 0.001, 0.01, 0.1, 1.')
print('Columns (left to right): C = 0.1, 1, 10, 100. Colour encodes test accuracy using the')
print('cividis colormap (light=high, dark=low). Numeric accuracy is printed in each cell.')
print('Top two rows are mostly light (high accuracy). Bottom row is uniformly dark (low accuracy).')


## 10. Doing It Properly With GridSearchCV

Sweeping manually over a grid is useful for visualisation, but in practice you'd use
`GridSearchCV`. It runs 5-fold cross-validation on the training set to find the best
hyperparameter combination. no test data involved, so there's no risk of data leakage.

The search space below matches the axes of Figure 5, so you can verify that GridSearchCV
finds the same sweet spot you saw in the heatmap.


In [ ]:
param_grid = {
    'C':     [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 'scale'],
}

gs = GridSearchCV(
    svm.SVC(kernel='rbf'),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
gs.fit(X_train_s, y_train)

print('Best parameters found:', gs.best_params_)
print('Cross-validated accuracy:', round(gs.best_score_, 3))
print('Test set accuracy:',        round(gs.best_estimator_.score(X_test_s, y_test), 3))


## 11. Final Evaluation

*(Figure 8)* Let's look at the full classification report and confusion matrix for the
best model returned by GridSearchCV.


In [ ]:
best = gs.best_estimator_
y_pred = best.predict(X_test_s)

print('Classification Report')
print('=' * 50)
print(classification_report(y_test, y_pred, target_names=wine.target_names))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=wine.target_names,
    cmap='Blues', ax=ax, colorbar=False
)
ax.set_title('Figure 8 — Confusion Matrix: Best RBF SVM', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig8_confusion_matrix.png', dpi=150)
plt.show()

print('Figure 8 alt-text: 3x3 confusion matrix. Rows are true cultivar labels')
print('(class_0, class_1, class_2 top to bottom); columns are predicted labels.')
print('Numbers on the diagonal are correct predictions; off-diagonal entries are errors.')
print('The matrix is nearly diagonal, indicating high accuracy across all three classes.')


---
## 12. What Makes a Valid Kernel? Mercer's Condition

So far we have used kernels as black boxes. But what actually *makes* a function a valid kernel?

A function K(x, x') is a valid kernel if and only if it satisfies **Mercer's condition**:
for any finite set of points {x₁, ..., xₙ}, the **Gram matrix** **K**  where K_ij = K(xᵢ, xⱼ)
must be **symmetric positive semi-definite (PSD)**.

Formally, for all vectors **c** ∈ ℝⁿ:

$$\sum_{i=1}^{n} \sum_{j=1}^{n} c_i c_j K(x_i, x_j) \geq 0$$

**Why does this matter practically?**

The SVM solves a convex quadratic programme. It is only guaranteed to have a unique global optimum
if the kernel (Gram) matrix is PSD. A non-PSD kernel can cause the optimiser to diverge or find
spurious solutions  your code may appear to run, but the results cannot be trusted.

**Intuition:** A PSD matrix can be written as AᵀA for some matrix A. This is exactly what a kernel
does implicitly  it computes inner products in a feature space Φ, so K(x, x') = Φ(x)·Φ(x').
The PSD condition is the algebraic fingerprint of a genuine inner product.

**How to check your own kernel:** compute the Gram matrix on a small subsample and verify that
all eigenvalues are ≥ 0. We will do exactly this in the next section.

> **Practical rule of thumb:** If you can derive your kernel from a known PSD kernel
> (by scaling, adding, or composing), it is PSD. If you're unsure, check eigenvalues
> numerically as shown below  it takes three lines of numpy.


## 13. Building a Custom Kernel From Scratch

The clearest way to demonstrate you understand kernels mathematically is to *implement one yourself*.

I will build a **Laplacian kernel** (exponential kernel with L1 distance):

$$K_{\\text{Laplacian}}(x, x') = exp left(-gamma |x - x'|_1right)$$

where ‖·‖₁ is the Manhattan distance. Compare this to RBF which uses L2 distance. This kernel is:
- **Mercer-valid** — proven via Bochner's theorem for L1 distance (Schölkopf & Smola, 2002)
- **More robust to outliers** — L1 grows linearly with distance, not quadratically
- **Not in scikit-learn** — a genuine custom implementation using `kernel='precomputed'`

When `kernel='precomputed'`, scikit-learn expects you to supply the full N×N kernel matrix. You are responsible for the maths; sklearn only does the quadratic optimisation.


In [ ]:
# implement the Laplacian kernel

def laplacian_kernel(X, Y, gamma=1.0):
    """
    Laplacian (L1-RBF) kernel: K[i,j] = exp(-gamma * ||X[i] - Y[j]||_1)

    Mercer-valid by Bochner's theorem applied to the L1 norm.
    Parameters
    ----------
    X : ndarray (n, d)
    Y : ndarray (m, d)
    gamma : float  -- bandwidth (higher = more local influence)
    Returns
    -------
    K : ndarray (n, m)
    """
    K = np.zeros((X.shape[0], Y.shape[0]))
    for i, xi in enumerate(X):
        for j, yj in enumerate(Y):
            K[i, j] = np.exp(-gamma * np.sum(np.abs(xi - yj)))
    return K


def laplacian_kernel_fast(X, Y, gamma=1.0):
    """
    Vectorised Laplacian kernel -- same result as the loop, ~100x faster.
    Uses numpy broadcasting to avoid Python loops.
    """
    diff = X[:, np.newaxis, :] - Y[np.newaxis, :, :]  # (n, m, d)
    return np.exp(-gamma * np.sum(np.abs(diff), axis=2))


# verify loop == vectorised
K_slow = laplacian_kernel(X_train_s[:5], X_train_s[:5], gamma=0.1)
K_fast = laplacian_kernel_fast(X_train_s[:5], X_train_s[:5], gamma=0.1)
print(f"Loop vs vectorised max diff: {np.abs(K_slow - K_fast).max():.2e}  (should be ~0)")


# verify Mercer's condition numerically
# A PSD matrix has all non-negative eigenvalues.
# We check on a 20-point subsample before trusting the full fit.
rng = np.random.default_rng(0)
idx = rng.choice(len(X_train_s), 20, replace=False)
K_check = laplacian_kernel_fast(X_train_s[idx], X_train_s[idx], gamma=0.1)
eigvals = np.linalg.eigvalsh(K_check)
print("\nGram matrix eigenvalues (all must be >= 0 for Mercer validity):")
print(np.round(eigvals, 6))
print(f"\nAll eigenvalues >= 0? {np.all(eigvals >= -1e-10)}")
print("Laplacian kernel passes the Mercer PSD check.")


In [ ]:
# tune gamma and C via manual cross-validation
# GridSearchCV does not support precomputed kernels with varying gamma
# because the kernel matrix changes with each gamma value.
# We handle this by recomputing the kernel matrix inside each CV fold.

from sklearn.model_selection import StratifiedKFold

gamma_candidates = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
C_candidates     = [1, 10, 100]
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_cv_score, best_params_lap = 0, {}

for gamma_val in gamma_candidates:
    for C_val in C_candidates:
        fold_scores = []
        for tr_idx, val_idx in cv5.split(X_train_s, y_train):
            Xtr,  Xval = X_train_s[tr_idx], X_train_s[val_idx]
            ytr,  yval = y_train[tr_idx],   y_train[val_idx]
            Ktr  = laplacian_kernel_fast(Xtr,  Xtr,  gamma=gamma_val)
            Kval = laplacian_kernel_fast(Xval, Xtr,  gamma=gamma_val)
            clf_tmp = svm.SVC(kernel="precomputed", C=C_val)
            clf_tmp.fit(Ktr, ytr)
            fold_scores.append(clf_tmp.score(Kval, yval))
        mean_score = float(np.mean(fold_scores))
        if mean_score > best_cv_score:
            best_cv_score  = mean_score
            best_params_lap = {"gamma": gamma_val, "C": C_val}

print(f"Best CV params : {best_params_lap}")
print(f"Best CV accuracy: {best_cv_score:.3f}")

# Refit on full training set with best params
g_best, C_best = best_params_lap["gamma"], best_params_lap["C"]
K_tr_best = laplacian_kernel_fast(X_train_s, X_train_s, gamma=g_best)
K_te_best = laplacian_kernel_fast(X_test_s,  X_train_s, gamma=g_best)
clf_lap   = svm.SVC(kernel="precomputed", C=C_best)
clf_lap.fit(K_tr_best, y_train)
lap_train_acc = clf_lap.score(K_tr_best, y_train)
lap_test_acc  = clf_lap.score(K_te_best,  y_test)
print(f"\nTuned Laplacian  train={lap_train_acc:.3f}  test={lap_test_acc:.3f}")


In [ ]:
# compare all kernels including the custom Laplacian
# Re-run the standard kernels so scores are in scope

std_configs = {
    "Linear":        svm.SVC(kernel="linear", C=1),
    "Poly (d=2)":    svm.SVC(kernel="poly", degree=2, C=1, coef0=1),
    "Poly (d=3)":    svm.SVC(kernel="poly", degree=3, C=1, coef0=1),
    "RBF (default)": svm.SVC(kernel="rbf",  C=1, gamma="scale"),
    "RBF (tuned)":   svm.SVC(kernel="rbf",  C=10, gamma=0.01),
}
std_results = []
for name, clf in std_configs.items():
    clf.fit(X_train_s, y_train)
    std_results.append({"Configuration": name,
                        "Train acc": clf.score(X_train_s, y_train),
                        "Test acc":  clf.score(X_test_s,  y_test)})

std_results.append({"Configuration": "Laplacian\n(custom)",
                    "Train acc": lap_train_acc,
                    "Test acc":  lap_test_acc})
all_df = pd.DataFrame(std_results)

x     = np.arange(len(all_df))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, all_df["Train acc"], width,
               label="Training accuracy", color=CB["blue"],   edgecolor="k", lw=0.6)
bars2 = ax.bar(x + width/2, all_df["Test acc"],  width,
               label="Test accuracy",     color=CB["orange"], edgecolor="k", lw=0.6)
ax.set_xticks(x)
ax.set_xticklabels(all_df["Configuration"], fontsize=9)
ax.set_ylim(0.70, 1.07)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Figure 7 — All Kernels Compared: Including Custom Laplacian",
             fontweight="bold", fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, linestyle="--", alpha=0.7)
ax.set_axisbelow(True)
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig("figures/fig7_all_kernels_comparison.png", dpi=150)
plt.show()

print("The custom Laplacian kernel is competitive with the built-in RBF,")
print("implemented from scratch using only numpy and the precomputed kernel API.")
print()
print("Figure 7 alt-text: Grouped bar chart with six kernel configurations on the x-axis")
print("(Linear, Poly d=2, Poly d=3, RBF default, RBF tuned, Laplacian custom).")
print("Blue bars = training accuracy, orange bars = test accuracy, values labelled on each bar.")
print("All configurations achieve >0.94 test accuracy. Train/test gaps are small for Linear")
print("and large for Poly d=3 (mild overfitting). The custom Laplacian matches tuned RBF.")


### Why L1 vs L2 Distance Matters

| | RBF (L2) | Laplacian (L1) |
|---|---|---|
| Distance formula | ‖x−x'‖₂ = √(Σ(xᵢ−xᵢ')²) | ‖x−x'‖₁ = Σ|xᵢ−xᵢ'| |
| Sensitivity to outliers | High (squares large differences) | Lower (linear growth) |
| Influence decay | Gaussian (rapid dropoff) | Exponential (slower dropoff) |
| Best when | Features are clean, continuous | Some features are noisy or irrelevant |

On the Wine dataset both kernels perform similarly because the features are clean and well-scaled. The Laplacian advantage becomes meaningful on noisier or sparser data for example, bag-of-words text representations where most dimensions are zero.

> **Key takeaway:** The kernel trick is not a fixed menu of three options. Any symmetric PSD function of two inputs is a valid kernel. Understanding Mercer's condition lets you design kernels tailored to your specific data structure.


---
## Summary: Custom Kernel vs Built-in Kernels

The table below summarises everything we've compared. The Laplacian kernel was built entirely
from scratch no scikit-learn kernel code, just numpy and the `precomputed` API.

| Kernel | Type | Needs tuning? | Best for |
|--------|------|--------------|----------|
| Linear | Built-in | C only | High-dimensional or near-linearly-separable data |
| Polynomial (d=2,3) | Built-in | C, degree, coef0 | Structured feature interactions |
| RBF | Built-in | C, gamma | Unknown structure; best default |
| Laplacian (L1-RBF) | **Custom** | C, gamma | Noisy/sparse features; outlier-robust |

On the Wine dataset all four perform similarly because the features are clean and well-scaled.
The Laplacian advantage becomes meaningful on noisier data (e.g. bag-of-words text)
where L1 distance is more natural than L2.

> **Key takeaway:** the kernel trick is not a fixed menu of three options.
> Any symmetric PSD function is a valid kernel. Mercer's condition is the gatekeeper.


## 14. What I Learned

A few things this exercise made concrete for me:

- **SVMs only care about the hard cases.** Once trained, all the 'easy' points are irrelevant the model depends entirely on the support vectors.
- **The kernel trick is genuinely clever.** You get the expressive power of high-dimensional space without ever constructing it explicitly.
- **Linear kernels are underrated.** On the wine data they matched the best-tuned RBF with zero effort. Don't skip the linear baseline.
- **RBF needs tuning.** With bad defaults it either retaining the training data (high gamma) or barely fits anything (low C). GridSearchCV is the right tool.
- **Standardise. Always.** This is the step that trips up most people new to SVMs.
- **You can write your own kernel.** Any symmetric PSD function is a valid kernel. The Laplacian kernel (L1-RBF) is not in scikit-learn but takes ~15 lines of numpy to implement and matches the built-in RBF on this dataset.
- **Mercer's condition is the gatekeeper.** Check that your custom kernel's Gram matrix has all non-negative eigenvalues before trusting your SVM results. A non-PSD kernel matrix means the optimiser has no convergence guarantee.

## References

Aeberhard, S., Coomans, D. and de Vel, O. (1992) Wine Data Set. UCI Machine Learning Repository. Available at: https://archive.ics.uci.edu/ml/datasets/wine (Accessed: 20 June 2026).

Cortes, C. and Vapnik, V. (1995) 'Support-vector networks', Machine Learning, 20(3), pp. 273–297. doi: 10.1007/BF00994018.

Hsu, C-W., Chang, C-C. and Lin, C-J. (2016) A practical guide to support vector classification. National Taiwan University. Available at: https://www.csie.ntu.edu.tw/~cjlin/papers/guide/guide.pdf (Accessed: 20 June 2026).

Nunez, J.R., Anderton, C.R. and Renslow, R.S. (2018) 'Optimizing colormaps with consideration for color vision deficiency', PLOS ONE, 13(7), e0199239. doi: 10.1371/journal.pone.0199239.

Pedregosa, F. et al. (2011) 'Scikit-learn: Machine learning in Python', Journal of Machine Learning Research, 12, pp. 2825–2830.

Scholkopf, B. and Smola, A.J. (2002) Learning with Kernels. Cambridge, MA: MIT Press.

Scikit-learn (2024) Support Vector Machines. Available at: https://scikit-learn.org/stable/modules/svm.html (Accessed: 20 June 2026).

Wong, B. (2011) 'Points of view: Color blindness', Nature Methods, 8(6), p. 441.

